In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_environment", "dev", "environment")
v_env = dbutils.widgets.get("p_environment")

In [0]:
from pyspark.sql.functions import col, lit, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

In [0]:
production_country_df = (
    spark.read
        .format("json")
        .option("multiline", "true")
        .load(f"{bronze_folder_path}/production_country")
    )
display(production_country_df)

countryId,movieId
161,62835
214,62837
214,62838
214,63020
214,63287
214,63492
214,63574
214,64328
214,64499
161,64559


In [0]:
production_country_shema = StructType([
    StructField("movieId", IntegerType()),
    StructField("countryId", IntegerType())
])

production_country_df = (
    spark.read
        .schema(production_country_shema)
        .format("json")
        .option("multiline", "true")
        .load(f"{bronze_folder_path}/production_country")
)
display(production_country_df)

movieId,countryId
62835,161
62837,214
62838,214
63020,214
63287,214
63492,214
63574,214
64328,214
64499,214
64559,161


In [0]:
production_country_df = (
    production_country_df
        .withColumnsRenamed(
            {
                "movieId": "movie_Id",
                "countryId": "country_id"
            }

        )
        .withColumns(
            {
                "env": lit(v_env)
            }
        )
)

movie_language_df = add_ingestion_date(production_country_df)

display(production_country_df)

movie_Id,country_id,env
62835,161,dev
62837,214,dev
62838,214,dev
63020,214,dev
63287,214,dev
63492,214,dev
63574,214,dev
64328,214,dev
64499,214,dev
64559,161,dev


In [0]:
(
    production_country_df
        .write
        .format("parquet")
        .mode("overwrite")
        .save(f"{silver_folder_path}/production_countries")
)

In [0]:
dbutils.notebook.exit("Success")